# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We leverage the Croissant metadata standard for flexible, FAIR data access and perform initial data wrangling and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL and loaded dynamically using `mlcroissant`.

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))

## 2. Data Overview
Review available record sets and their fields, using their Croissant `@id`s. This helps us know what data tables are present and what their structure is.

We will print a summary of each RecordSet, including the `@id` and fields.

In [ ]:
print("Record Sets available in this dataset:\n")
record_sets = []
for rset in getattr(metadata, 'record_sets', []):
    print(f"- @id: {rset.id}")
    print(f"  name: {getattr(rset, 'name', rset.id)}")
    print(f"  description: {getattr(rset, 'description', '<none>')}")
    field_ids = [field.id for field in getattr(rset, 'fields', [])]
    print(f"  fields: {field_ids}\n")
    record_sets.append(rset.id)

if not record_sets:
    # fallback: sometimes Croissant metadata puts all data in one RecordSet
    print("No explicit RecordSets found in metadata. Trying to infer RecordSets from dataset...")
    # Try to find record set IDs via dataset utility, fallback to default
    # NOTE: mlcroissant >=0.8.0 provides dataset.record_sets
    try:
        record_sets = dataset.record_sets
    except Exception:
        record_sets = ['cr:main']  # Replace with main record set if known
    print("RecordSets:", record_sets)

## 3. Data Extraction
Load data from a chosen record set into a DataFrame for analysis. Use the RecordSet and field `@id`s.

List available record set IDs and extract records for each.

In [ ]:
# Choose record set(s): use discovered or documented IDs
selected_record_sets = record_sets if len(record_sets) else ['cr:main']
dataframes = {}

for record_set_id in selected_record_sets:
    print(f"\nLoading records from RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Example columns: {df.columns.tolist()}")
        display(df.head(3))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick a record set to analyze
main_rs = selected_record_sets[0] if selected_record_sets else None

## 4. Exploratory Data Analysis (EDA)
Let's perform common EDA and data processing steps,
such as filtering by a numeric field, normalization and grouping by a key attribute.

We'll look for likely quantitative fields (e.g. abundance, MW) for demonstration.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = dataframes.get(main_rs)
numeric_candidates = []
if df is not None:
    for col in df.columns:
        try:
            vals = pd.to_numeric(df[col], errors='coerce')
            nan_frac = np.mean(vals.isna())
            if nan_frac < 0.5 and vals.dtype in ['float64', 'int64']:
                numeric_candidates.append(col)
        except Exception:
            continue

    print(f"Numeric columns inferred: {numeric_candidates}")
    # Pick one numeric field to demonstrate (e.g. abundance, MW, Coverage)
    preferred_numeric = None
    for field in ['Abundance', 'MW', 'Coverage', 'PeptideCount', 'SumPsm', 'Intensity', 'QuantValue']:
        if field in df.columns:
            preferred_numeric = field
            break
    if not preferred_numeric:
        if numeric_candidates:
            preferred_numeric = numeric_candidates[0]

    # Attempt filtering if a numeric field was found
    if preferred_numeric:
        numeric_field = preferred_numeric
        print(f"\nProcessing numeric field: '{numeric_field}'")
        # Convert to numeric as necessary
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75) if len(df) > 40 else df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered {len(filtered_df)}/{len(df)} records with {numeric_field} > {threshold:.2f}.")
        display(filtered_df.head(3))

        # Normalize the selected numeric field
        filtered_df[numeric_field + '_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized '{numeric_field}':")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head(3))

        # Attempt to group by a protein or categorical field
        group_field = None
        for col in ['Protein', 'Gene', 'Accession', 'Description', 'Group']:
            if col in df.columns:
                group_field = col
                break
        if not group_field:
            # fallback: first object dtype field
            object_cols = df.select_dtypes(include='object').columns.tolist()
            group_field = object_cols[0] if object_cols else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nMean of {numeric_field} grouped by {group_field}: (Top 5)")
            display(grouped_df.head(5))
    else:
        print("No suitable numeric field found for demo EDA.")
else:
    print("No DataFrame for RecordSet.")

## 5. Visualization
Let's visualize the distribution of a selected numeric field and its normalized values. 

We'll plot the histogram and, if possible, boxplot for the numeric field used in EDA.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and preferred_numeric:
    plt.figure(figsize=(10,4))
    sns.histplot(df[preferred_numeric].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of '{preferred_numeric}'")
    plt.xlabel(preferred_numeric)
    plt.ylabel('Count')
    plt.show()

    if filtered_df is not None and (preferred_numeric + '_normalized') in filtered_df.columns:
        plt.figure(figsize=(8, 3))
        sns.boxplot(x=filtered_df[preferred_numeric + '_normalized'], fliersize=2, color="lightgreen")
        plt.title(f"Boxplot of normalized '{preferred_numeric}' (filtered)")
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² mass spectrometry protein dataset for human mast cell extracellular vesicles using `mlcroissant`. We explored available record sets and their structure, loaded records into DataFrames, and performed filtering, normalization, grouping, and simple visualization on selected quantitative fields such as protein abundance or molecular weight. This approach can be extended further for advanced analysis or modeling workflows.

**Tip**: To further customize the EDA, inspect the DataFrame columns in section 3 and adjust filtering/grouping columns as needed for your own use case.